# Лабораторная работа №4: Self-supervised learning визуальных представлений

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Извлечь frozen DINO embeddings.
- Сравнить k-NN и linear probe при 1/10/100% меток.
- Исследовать ближайших соседей и shortcut features.


## Теоретическая часть

Self-supervised encoder обучается без классов: contrastive objective сближает представления двух views одного изображения, masked modeling восстанавливает скрытые токены. Качество representation проверяют frozen k-NN и linear probe.

![DINO](https://raw.githubusercontent.com/facebookresearch/dino/main/.github/dino.gif)


### Математическая постановка

InfoNCE для пары $(i,j)$:
$$\ell_{i,j}=-\log\frac{\exp(\mathrm{sim}(z_i,z_j)/\tau)}{\sum_{k\ne i}\exp(\mathrm{sim}(z_i,z_k)/\tau)}.$$
Linear probe обучает только $W$ в $\hat y=\arg\max Wz$.


### Материалы

- [DINOv2](https://github.com/facebookresearch/dinov2)
- [SimCLR](https://arxiv.org/abs/2002.05709)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Извлечь frozen DINO embeddings.
2. Сравнить k-NN и linear probe при 1/10/100% меток.
3. Исследовать ближайших соседей и shortcut features.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** DINOv2 ViT-S/14 frozen, FP16.
- **Ограничение данных:** до 5 000 изображений.
- **Fallback:** ResNet18 frozen embeddings.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [1]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    # локально работаю на Apple Silicon, поэтому кроме CUDA проверяю MPS
    DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})

{'device': 'mps', 'fast_dev_run': False, 'data_root': '/Users/filipp/Desktop/рабочий стол/Вуз/учеба/computer_vision/computer_vision_2026/data'}


In [2]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Стратифицированные разбиения CIFAR-10

*Вес: 1 балл.*

**Постановка.** Train, validation и test изолируют подбор параметров линейного probe.

Выполните следующие действия:
1. Загрузите CIFAR-10 и стратифицируйте train.
2. Подготовьте неизменяемые индексы test.

**Результат.** Датасет ds, загрузчик loader и длины трёх выборок.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [3]:
import torchvision
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# профиль работы — до 5 000 изображений: 3 000 train + 1 000 validation из официального train и 1 000 из официального test
N_TRAIN, N_VAL, N_TEST = (300, 100, 100) if FAST_DEV_RUN else (3000, 1000, 1000)
raw_train = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True)
raw_test = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True)
classes = raw_train.classes
y_tr_all, y_te_all = np.array(raw_train.targets), np.array(raw_test.targets)

pool, _ = train_test_split(np.arange(len(y_tr_all)), train_size=N_TRAIN + N_VAL, stratify=y_tr_all, random_state=SEED)
train_idx, val_idx = train_test_split(pool, test_size=N_VAL, stratify=y_tr_all[pool], random_state=SEED)
test_idx, _ = train_test_split(np.arange(len(y_te_all)), train_size=N_TEST, stratify=y_te_all, random_state=SEED)
train_idx, val_idx, test_idx = np.sort(train_idx), np.sort(val_idx), np.sort(test_idx)   # неизменяемые индексы

# один тензор со всеми выбранными изображениями: сначала train, затем validation, затем test
images = torch.from_numpy(np.concatenate([raw_train.data[train_idx], raw_train.data[val_idx], raw_test.data[test_idx]])).permute(0, 3, 1, 2)
labels = np.concatenate([y_tr_all[train_idx], y_tr_all[val_idx], y_te_all[test_idx]])
split = np.array(["train"] * len(train_idx) + ["val"] * len(val_idx) + ["test"] * len(test_idx))
ds = TensorDataset(images, torch.from_numpy(labels))
loader = DataLoader(ds, batch_size=128, shuffle=False)
y_train, y_val, y_test = labels[split == "train"], labels[split == "val"], labels[split == "test"]
check(len(set(train_idx) & set(val_idx)) == 0, "train и validation не пересекаются")
check(all(np.bincount(y, minlength=10).min() == len(y) // 10 for y in (y_train, y_val, y_test)), "все три выборки сбалансированы")
print({"train": len(y_train), "val": len(y_val), "test": len(y_test)})

✔ train и validation не пересекаются
✔ все три выборки сбалансированы
{'train': 3000, 'val': 1000, 'test': 1000}


### Предобученный DINOv2 и признаки

*Вес: 1 балл.*

**Постановка.** Frozen encoder предоставляет представления без доступа к меткам при извлечении.

Выполните следующие действия:
1. Загрузите DINOv2 ViT-S/14.
2. Сохраните нормированные признаки и явные train/validation/test индексы.

**Результат.** embeddings.npz с фактическими DINO-признаками.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [4]:
import torch.nn.functional as F
import transformers
from transformers import AutoModel

DINO_ID = "facebook/dinov2-small"   # ViT-S/14
IMG_SIZE = 224
encoder = AutoModel.from_pretrained(DINO_ID).to(DEVICE).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def prepare(x_uint8):
    x = x_uint8.to(DEVICE).float() / 255
    x = F.interpolate(x, size=IMG_SIZE, mode="bicubic", align_corners=False).clamp(0, 1)
    return (x - MEAN) / STD

@torch.no_grad()
def extract(enc, data_loader):
    enc.eval()
    feats = [F.normalize(enc(pixel_values=prepare(x)).pooler_output, dim=1).cpu() for x, _ in data_loader]
    return torch.cat(feats).numpy()

t0 = time.time()
X = extract(encoder, loader)    # CLS-эмбеддинг после layernorm, L2-нормированный; метки при извлечении не нужны
extract_time = time.time() - t0
np.savez_compressed(ARTIFACTS / "embeddings.npz", features=X.astype(np.float16), labels=labels, split=split,
                    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx, model=DINO_ID)
X_train, X_val, X_test = X[split == "train"], X[split == "val"], X[split == "test"]
check(np.allclose(np.linalg.norm(X, axis=1), 1, atol=1e-3), "признаки L2-нормированы")
print(f"{DINO_ID}: признаки {X.shape}, {len(X) / extract_time:.0f} изобр./с на {DEVICE}")

✔ признаки L2-нормированы
facebook/dinov2-small: признаки (5000, 384), 416 изобр./с на mps


### Подбор k-NN на validation

*Вес: 1 балл.*

**Постановка.** Число соседей следует выбирать без обращения к test.

Выполните следующие действия:
1. Обучите k-NN для предложенной сетки k.
2. Выберите k по validation accuracy.

**Результат.** Словарь knn_val и выбранный best_k.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [5]:
from sklearn.neighbors import KNeighborsClassifier
K_GRID = [1, 3, 5, 10, 20, 50]
knn_val = {}
for k in K_GRID:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", weights="distance").fit(X_train, y_train)
    knn_val[k] = float((knn.predict(X_val) == y_val).mean())
best_k = max(knn_val, key=knn_val.get)
print("validation accuracy k-NN:", knn_val, "→ best_k =", best_k)

validation accuracy k-NN: {1: 0.95, 3: 0.962, 5: 0.965, 10: 0.965, 20: 0.962, 50: 0.964} → best_k = 5


### Линейная проверка и test

*Вес: 1 балл.*

**Постановка.** Frozen-признаки оценивают независимыми простыми классификаторами.

Выполните следующие действия:
1. Обучите k-NN и logistic regression только на train.
2. Вычислите test accuracy и macro-F1 после выбора k.

**Результат.** probes.json с измеренными test-метриками.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def fit_linear(Xtr, ytr, C):
    return LogisticRegression(C=C, max_iter=3000).fit(Xtr, ytr)

C_GRID = [0.1, 1.0, 10.0, 100.0]
val_lin = {C: float(fit_linear(X_train, y_train, C).score(X_val, y_val)) for C in C_GRID}
best_C = max(val_lin, key=val_lin.get)
knn = KNeighborsClassifier(n_neighbors=best_k, metric="cosine", weights="distance").fit(X_train, y_train)
linear = fit_linear(X_train, y_train, best_C)
probes = {}
for name, clf in [("knn", knn), ("linear", linear)]:
    pred = clf.predict(X_test)
    probes[name] = {"test_accuracy": float(accuracy_score(y_test, pred)), "test_macro_f1": float(f1_score(y_test, pred, average="macro"))}
probes["selected_on_validation"] = {"k": best_k, "knn_val": knn_val, "C": best_C, "linear_val": val_lin}
probes["n"] = {"train": len(y_train), "val": len(y_val), "test": len(y_test)}
(ARTIFACTS / "probes.json").write_text(json.dumps(probes, indent=2))
print(json.dumps({k: probes[k] for k in ("knn", "linear")}, indent=1), "| C =", best_C)

{
 "knn": {
  "test_accuracy": 0.952,
  "test_macro_f1": 0.9518996271403667
 },
 "linear": {
  "test_accuracy": 0.959,
  "test_macro_f1": 0.959018757538872
 }
} | C = 10.0


### Кривая объёма разметки

*Вес: 1 балл.*

**Постановка.** Малые размеченные подмножества выявляют sample efficiency представления.

Выполните следующие действия:
1. Создайте сбалансированные подмножества train по фиксированному seed.
2. Обучите отдельный linear probe для каждой доли и сохраните результаты.

**Результат.** label_curve.json с числом объектов и измеренной точностью.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [7]:
def balanced_subset(y, fraction, seed):
    rng = np.random.RandomState(seed)
    per_class = max(1, int(round(fraction * len(y) / 10)))
    return np.concatenate([rng.choice(np.where(y == c)[0], per_class, replace=False) for c in range(10)])

FRACTIONS = [0.01, 0.10, 1.00]
SEEDS = [0, 1, 2, 3, 4]
label_curve = []
for frac in FRACTIONS:
    accs = []
    for s in (SEEDS if frac < 1 else [0]):
        sub = balanced_subset(y_train, frac, SEED + s)
        accs.append(float(fit_linear(X_train[sub], y_train[sub], best_C).score(X_test, y_test)))
    label_curve.append({"fraction": frac, "n_labeled": len(balanced_subset(y_train, frac, SEED)), "seeds": len(accs),
                        "test_accuracy_mean": float(np.mean(accs)), "test_accuracy_std": float(np.std(accs)), "runs": accs})
(ARTIFACTS / "label_curve.json").write_text(json.dumps({"probe": f"logreg C={best_C}", "curve": label_curve}, indent=2))
for r in label_curve:
    print(f"{r['fraction']:>5.0%} ({r['n_labeled']:4d} меток): test acc {r['test_accuracy_mean']:.4f} ± {r['test_accuracy_std']:.4f}")

   1% (  30 меток): test acc 0.8462 ± 0.0220
  10% ( 300 меток): test acc 0.9378 ± 0.0080
 100% (3000 меток): test acc 0.9590 ± 0.0000


### Контрастивная адаптация encoder

*Вес: 1 балл.*

**Постановка.** Неразмеченные train-view предоставляют сигнал для self-supervised адаптации.

Выполните следующие действия:
1. Создайте две аугментации одних train-изображений.
2. Оптимизируйте InfoNCE и сохраните обновлённый encoder.

**Результат.** encoder_adapted.pt после реальных optimizer steps.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [8]:
import copy, torch.nn as nn
from torchvision.transforms import v2 as T2

SSL_EPOCHS = 1 if FAST_DEV_RUN else 8
SSL_BATCH, TAU, SSL_LR = 128, 0.2, 2e-5
N_UNFROZEN = 4   # дообучаю только последние блоки ViT, ранние слои заморожены — меньше риск разрушить представление

adapted = copy.deepcopy(encoder).to(DEVICE)
for p in adapted.parameters():
    p.requires_grad = False
for blk in adapted.encoder.layer[-N_UNFROZEN:]:
    for p in blk.parameters():
        p.requires_grad = True
for p in adapted.layernorm.parameters():
    p.requires_grad = True
proj = nn.Sequential(nn.Linear(384, 512), nn.ReLU(), nn.Linear(512, 128)).to(DEVICE)
augment = T2.Compose([T2.RandomResizedCrop(32, scale=(0.4, 1.0), antialias=True), T2.RandomHorizontalFlip(),
                      T2.RandomApply([T2.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8), T2.RandomGrayscale(0.2)])

def info_nce(z1, z2, tau=TAU):
    """NT-Xent: позитив — другой view того же изображения, негативы — остальные 2N-2 view в батче."""
    z = F.normalize(torch.cat([z1, z2]), dim=1)
    sim = z @ z.T / tau
    sim.fill_diagonal_(float("-inf"))
    n = len(z1)
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)

train_images = images[split == "train"]       # метки на этом шаге не используются
ssl_loader = DataLoader(TensorDataset(train_images), batch_size=SSL_BATCH, shuffle=True, drop_last=True,
                        generator=torch.Generator().manual_seed(SEED))
params = [p for p in adapted.parameters() if p.requires_grad] + list(proj.parameters())
opt = torch.optim.AdamW(params, lr=SSL_LR, weight_decay=0.05)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, SSL_EPOCHS * len(ssl_loader))
ssl_hist, steps = [], 0
torch.manual_seed(SEED)
t0 = time.time()
for epoch in range(SSL_EPOCHS):
    adapted.train(); losses = []
    for (x,) in ssl_loader:
        v1 = torch.stack([augment(img) for img in x]); v2 = torch.stack([augment(img) for img in x])
        z1 = proj(adapted(pixel_values=prepare(v1)).pooler_output)
        z2 = proj(adapted(pixel_values=prepare(v2)).pooler_output)
        loss = info_nce(z1, z2)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item()); steps += 1
    ssl_hist.append(float(np.mean(losses)))
    print(f"InfoNCE эпоха {epoch + 1}/{SSL_EPOCHS}: {ssl_hist[-1]:.4f} ({time.time() - t0:.0f} c)")
torch.save({"encoder": adapted.state_dict(), "projection": proj.state_dict(), "base_model": DINO_ID, "optimizer_steps": steps,
            "epochs": SSL_EPOCHS, "lr": SSL_LR, "tau": TAU, "unfrozen_blocks": N_UNFROZEN, "loss_history": ssl_hist},
           ARTIFACTS / "encoder_adapted.pt")
check(steps > 0 and np.isfinite(ssl_hist).all(), f"выполнено {steps} реальных optimizer steps, loss конечен")

InfoNCE эпоха 1/8: 2.6801 (29 c)


InfoNCE эпоха 2/8: 2.1797 (57 c)


InfoNCE эпоха 3/8: 2.0434 (85 c)


InfoNCE эпоха 4/8: 1.9498 (113 c)


InfoNCE эпоха 5/8: 1.9387 (141 c)


InfoNCE эпоха 6/8: 1.8914 (168 c)


InfoNCE эпоха 7/8: 1.8753 (196 c)


InfoNCE эпоха 8/8: 1.8954 (224 c)


✔ выполнено 184 реальных optimizer steps, loss конечен


### Признаки после адаптации

*Вес: 1 балл.*

**Постановка.** Сравнение до и после адаптации требует одних и тех же изображений.

Выполните следующие действия:
1. Пересчитайте признаки train/validation/test обновлённым encoder.
2. Обучите новый linear probe на train.

**Результат.** Матрица X_new и обученный new_linear.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [9]:
X_new = extract(adapted, loader)
Xn_train, Xn_val, Xn_test = X_new[split == "train"], X_new[split == "val"], X_new[split == "test"]
val_new = {C: float(fit_linear(Xn_train, y_train, C).score(Xn_val, y_val)) for C in C_GRID}
best_C_new = max(val_new, key=val_new.get)
new_linear = fit_linear(Xn_train, y_train, best_C_new)
print("validation accuracy linear probe после адаптации:", val_new, "→ C =", best_C_new)

validation accuracy linear probe после адаптации: {0.1: 0.931, 1.0: 0.958, 10.0: 0.97, 100.0: 0.974} → C = 100.0


### Точность адаптированного probe

*Вес: 1 балл.*

**Постановка.** Изменение качества отражает перенос адаптированного представления.

Выполните следующие действия:
1. Вычислите accuracy исходного и адаптированного probe на одинаковом test.
2. Сохраните парный результат.

**Результат.** adaptation.json с наблюдаемыми показателями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [10]:
from scipy.stats import binomtest
pred_old, pred_new = linear.predict(X_test), new_linear.predict(Xn_test)
ok_old, ok_new = pred_old == y_test, pred_new == y_test
table = [[int((ok_old & ok_new).sum()), int((ok_old & ~ok_new).sum())], [int((~ok_old & ok_new).sum()), int((~ok_old & ~ok_new).sum())]]
adaptation = {"n_test": len(y_test), "original": {"test_accuracy": float(ok_old.mean()), "test_macro_f1": float(f1_score(y_test, pred_old, average="macro"))},
              "adapted": {"test_accuracy": float(ok_new.mean()), "test_macro_f1": float(f1_score(y_test, pred_new, average="macro"))},
              "delta_accuracy": float(ok_new.mean() - ok_old.mean()), "mcnemar_table": table,
              "mcnemar_exact_p": float(binomtest(table[0][1], table[0][1] + table[1][0], 0.5).pvalue) if table[0][1] + table[1][0] else 1.0, "ssl": {"epochs": SSL_EPOCHS, "steps": steps, "lr": SSL_LR}}
(ARTIFACTS / "adaptation.json").write_text(json.dumps(adaptation, indent=2))
print(json.dumps(adaptation, indent=1))

{
 "n_test": 1000,
 "original": {
  "test_accuracy": 0.959,
  "test_macro_f1": 0.959018757538872
 },
 "adapted": {
  "test_accuracy": 0.957,
  "test_macro_f1": 0.9571113953681785
 },
 "delta_accuracy": -0.0020000000000000018,
 "mcnemar_table": [
  [
   937,
   22
  ],
  [
   20,
   21
  ]
 ],
 "mcnemar_exact_p": 0.8776143287523154,
 "ssl": {
  "epochs": 8,
  "steps": 184,
  "lr": 2e-05
 }
}


### Визуальные соседи

*Вес: 1 балл.*

**Постановка.** Ранжирование соседей позволяет выявить смешение категорий.

Выполните следующие действия:
1. Найдите ближайшие train-объекты для фиксированных test-запросов.
2. Сохраните расстояния, индексы и метки.

**Результат.** neighbors.json с фактическими близкими изображениями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Реализуйте это задание по постановке выше.
raise NotImplementedError("Реализуйте это задание в этой ячейке")


### Чистота ближайшего окружения

*Вес: 1 балл.*

**Постановка.** Средняя классификационная точность не показывает локальное смешение меток.

Выполните следующие действия:
1. По уже вычисленным пяти соседям найдите долю совпадений классов.
2. Выведите среднюю чистоту и примеры с минимальной чистотой.

**Результат.** neighborhood_purity.json с измерениями на test-запросах.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Реализуйте это задание по постановке выше.
raise NotImplementedError("Реализуйте это задание в этой ячейке")


### Творческий бонус: ошибки соседства

*Вес: 1 дополнительный балл.*

**Постановка.** Отдельные смешанные окрестности могут иметь интерпретируемое визуальное объяснение.

Выполните следующие действия:
1. Выберите два test-изображения с низкой чистотой соседей.
2. Сохраните коллаж запросов и соответствующих train-соседей.

**Результат.** bonus_neighbors.png и индексы изображений.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# Реализуйте это задание по постановке выше.
raise NotImplementedError("Реализуйте это задание в этой ячейке")


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.
